In [1]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne
from scipy.linalg import logm, expm
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score
from pyriemann.estimation import Covariances
from sklearn.decomposition import PCA
from pyriemann.utils.mean import mean_logeuclid
from pyriemann.utils.tangentspace import tangent_space

In [2]:
def compute_enhanced_cst_pca(bar_s_source, bar_t_target_align, y_source, y_target_align, classes, n_clusters=3):
    """
    Enhance Cst computation using PCA-based clustering as described in section 2.4 of the paper.
    For each class:
    1. Train PCA on source data
    2. Project both source and target data onto PCA space
    3. For each principal component, create clusters based on component values
    4. Compute mean of each cluster as anchor points
    """
    
    # Lists to store all anchor points
    s_anchors = []
    t_anchors = []
    
    # For each class, create clusters using PCA
    for cls in classes:
        # Get data for this class
        s_cls = bar_s_source[y_source == cls]
        t_cls = bar_t_target_align[y_target_align == cls]
        
        if len(s_cls) < n_clusters or len(t_cls) < n_clusters:
            # If not enough samples, just use class mean
            s_anchors.append(np.mean(s_cls, axis=0))
            t_anchors.append(np.mean(t_cls, axis=0))
            continue
        
        # Train PCA on source data (just first component as shown in Figure 2)
        pca = PCA(n_components=1)
        pca.fit(s_cls)
        
        # Project both source and target data
        s_proj = pca.transform(s_cls).flatten()
        t_proj = pca.transform(t_cls).flatten()
        
        # Sort by PCA scores
        s_sorted_idx = np.argsort(s_proj)
        t_sorted_idx = np.argsort(t_proj)
        
        # Split into n_clusters clusters along the first principal component
        s_chunk_size = len(s_sorted_idx) // n_clusters
        t_chunk_size = len(t_sorted_idx) // n_clusters
        
        # Create clusters and compute means
        for i in range(n_clusters):
            s_start = i * s_chunk_size
            s_end = None if i == n_clusters-1 else (i+1) * s_chunk_size
            s_cluster_idx = s_sorted_idx[s_start:s_end]
            
            t_start = i * t_chunk_size
            t_end = None if i == n_clusters-1 else (i+1) * t_chunk_size
            t_cluster_idx = t_sorted_idx[t_start:t_end]
            
            # Compute and store means for this cluster
            s_anchors.append(np.mean(s_cls[s_cluster_idx], axis=0))
            t_anchors.append(np.mean(t_cls[t_cluster_idx], axis=0))
    
    # Stack all anchor points
    bar_S = np.column_stack(s_anchors)  # Shape: (n_features, n_anchors)
    bar_T = np.column_stack(t_anchors)  # Shape: (n_features, n_anchors)
    
    # Compute cross-product matrix (as in the paper)
    C_st = bar_S @ bar_T.T
    
    return C_st, bar_S, bar_T


In [3]:
def encode_labels(labels_list):
    encoded_list = []
    for labels in labels_list:
        # Create mapping from original labels to 0,1
        unique_labels = np.unique(labels)
        label_map = {unique_labels[i]: i for i in range(len(unique_labels))}
        
        # Apply mapping
        encoded = np.array([label_map[label] for label in labels])
        encoded_list.append(encoded)
    return encoded_list

In [4]:
# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=5):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data

In [5]:
pairs = [
    (0.5, 3.5),
    (0.5, 2.5),
    (0.5, 1.5),
    (1.5, 3.5),
    (1.5, 2.5),
    (2.5, 3.5)
]


In [6]:

for tmin, tmax in pairs:
    print(f"tmin = {tmin}, tmax = {tmax}")

tmin = 0.5, tmax = 3.5
tmin = 0.5, tmax = 2.5
tmin = 0.5, tmax = 1.5
tmin = 1.5, tmax = 3.5
tmin = 1.5, tmax = 2.5
tmin = 2.5, tmax = 3.5


In [7]:
def twofour_crosssession(n_classes):
    for tmin, tmax in pairs:     
        data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCI2b/BCICIV_2b_gdf'

        # Lists to hold data for all subjects
        train_active_X = []         # List to hold numpy arrays with shape (n_trials, n_channels, n_times) per subject
        train_active_y = []         # List to hold event labels per subject
        train_active_metadata = []  # List to hold event metadata per subject

        # Define subject IDs (B01 to B09)
        subjects = [f'B{subj:02d}' for subj in range(1, 10)]

        for subj in subjects:
            # Define training sessions for this subject (e.g., B0101T.gdf, B0102T.gdf, B0103T.gdf)
            session_ids = ['01T'] #, '02T', '03T']
            subj_epochs_list = []

            for sess in session_ids:
                filename = os.path.join(data_dir, f'{subj}{sess}.gdf')
                
                # Check if file exists to avoid errors
                if not os.path.exists(filename):
                    print(f"File {filename} not found, skipping.")
                    continue
                
                # Load the GDF file
                raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)
                
                # Extract events from annotations, mapping '769' to 1 (left) and '770' to 2 (right)
                event_id_mapping = {'769': 1, '770': 2}
                events, event_dict = mne.events_from_annotations(raw, event_id=event_id_mapping, verbose=False)
                
                # Select only EEG channels (C3, Cz, C4)
                print(raw.ch_names)
                eeg_channels = [ch for ch in raw.ch_names if ch in ['EEG:C3', 'EEG:Cz', 'EEG:C4']]
                if len(eeg_channels) != 3:
                    print(f"Warning: Expected 3 EEG channels for {subj}{sess}, found {len(eeg_channels)}: {eeg_channels}")
                raw_eeg = raw.pick_channels(eeg_channels, verbose=False)
                
                # Create epochs
                epochs = mne.Epochs(
                    raw_eeg,
                    events,
                    event_id={'left': 1, 'right': 2},
                    tmin=tmin,
                    tmax=tmax,
                    baseline=None,  # No baseline correction, matching your 2a code
                    preload=True,
                    verbose=False
                )
                
                subj_epochs_list.append(epochs)
            
            # Skip subject if no sessions were processed
            if not subj_epochs_list:
                print(f"No valid sessions found for subject {subj}, skipping.")
                continue
            
            # Concatenate epochs across sessions for this subject
            if len(subj_epochs_list) > 1:
                subj_epochs = mne.concatenate_epochs(subj_epochs_list, verbose=False)
            else:
                subj_epochs = subj_epochs_list[0]
            
            # Get the epoch data
            subj_data = subj_epochs.get_data()
            
            # Apply causal bandpass filter to each trial and channel (matching Dataset 2a)
            n_trials, n_channels, n_times = subj_data.shape
            fs = raw.info['sfreq']  # Sampling frequency (250 Hz for Dataset 2b)
            subj_filtered_data = np.empty_like(subj_data)
            for trial in range(n_trials):
                for ch in range(n_channels):
                    subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                        subj_data[trial, ch, :],
                        lowcut=8,   # Lower bound of sensorimotor rhythm
                        highcut=30, # Upper bound of sensorimotor rhythm
                        fs=fs,
                        order=50    # Filter order
                    )
            
            # Append processed data, labels, and metadata
            train_active_X.append(subj_filtered_data)
            train_active_y.append(subj_epochs.events[:, 2])  # Labels in third column (1 or 2)
            train_active_metadata.append(subj_epochs.events)
            
            # Print shape to verify
            print(f"Subject {subj}: Epoch data shape {subj_filtered_data.shape}")

        print(f"Loaded data for {len(train_active_X)} subjects.")


        # Lists to hold data for all subjects
        eval_active_X = []         # List to hold numpy arrays with shape (n_trials, n_channels, n_times) per subject
        eval_active_y = []         # List to hold event labels per subject
        eval_active_metadata = []  # List to hold event metadata per subject

        # Define subject IDs (B01 to B09)
        subjects = [f'B{subj:02d}' for subj in range(1, 10)]

        for subj in subjects:
            # Define training sessions for this subject (e.g., B0101T.gdf, B0102T.gdf, B0103T.gdf)
            session_ids = ['02T'] #, '02T', '03T']
            subj_epochs_list = []

            for sess in session_ids:
                filename = os.path.join(data_dir, f'{subj}{sess}.gdf')
                
                # Check if file exists to avoid errors
                if not os.path.exists(filename):
                    print(f"File {filename} not found, skipping.")
                    continue
                
                # Load the GDF file
                raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)
                
                # Extract events from annotations, mapping '769' to 1 (left) and '770' to 2 (right)
                event_id_mapping = {'769': 1, '770': 2}
                events, event_dict = mne.events_from_annotations(raw, event_id=event_id_mapping, verbose=False)
                
                # Select only EEG channels (C3, Cz, C4)
                print(raw.ch_names)
                eeg_channels = [ch for ch in raw.ch_names if ch in ['EEG:C3', 'EEG:Cz', 'EEG:C4']]
                if len(eeg_channels) != 3:
                    print(f"Warning: Expected 3 EEG channels for {subj}{sess}, found {len(eeg_channels)}: {eeg_channels}")
                raw_eeg = raw.pick_channels(eeg_channels, verbose=False)
                
                
                # Create epochs
                epochs = mne.Epochs(
                    raw_eeg,
                    events,
                    event_id={'left': 1, 'right': 2},
                    tmin=tmin,
                    tmax=tmax,
                    baseline=None,  # No baseline correction, matching your 2a code
                    preload=True,
                    verbose=False
                )
                
                subj_epochs_list.append(epochs)
            
            # Skip subject if no sessions were processed
            if not subj_epochs_list:
                print(f"No valid sessions found for subject {subj}, skipping.")
                continue
            
            # Concatenate epochs across sessions for this subject
            if len(subj_epochs_list) > 1:
                subj_epochs = mne.concatenate_epochs(subj_epochs_list, verbose=False)
            else:
                subj_epochs = subj_epochs_list[0]
            
            # Get the epoch data
            subj_data = subj_epochs.get_data()
            
            # Apply causal bandpass filter to each trial and channel (matching Dataset 2a)
            n_trials, n_channels, n_times = subj_data.shape
            fs = raw.info['sfreq']  # Sampling frequency (250 Hz for Dataset 2b)
            subj_filtered_data = np.empty_like(subj_data)
            for trial in range(n_trials):
                for ch in range(n_channels):
                    subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                        subj_data[trial, ch, :],
                        lowcut=8,   # Lower bound of sensorimotor rhythm
                        highcut=30, # Upper bound of sensorimotor rhythm
                        fs=fs,
                        order=50    # Filter order
                    )
            
            # Append processed data, labels, and metadata
            eval_active_X.append(subj_filtered_data)
            eval_active_y.append(subj_epochs.events[:, 2])  # Labels in third column (1 or 2)
            eval_active_metadata.append(subj_epochs.events)
            
            # Print shape to verify
            print(f"Subject {subj}: Epoch data shape {subj_filtered_data.shape}")

        print(f"Loaded data for {len(train_active_X)} subjects.")
        
        train_active_y = encode_labels(train_active_y)
        eval_active_y = encode_labels(eval_active_y)

        if(n_classes==2):
            classes = [0, 1]
        else:
            classes = [0, 1, 2, 3]

        align_per_class = 12
        n_subjects = 9
        
        accuracies = []

        for subj_idx in range(len(train_active_X)):
            # print(subj_idx)
            # Split data into train/test using leave-one-subject-out
            X_target = eval_active_X[subj_idx]
            y_target = eval_active_y[subj_idx]
            
            # Concatenate data from other subjects
            X_source = train_active_X[subj_idx]
            y_source = train_active_y[subj_idx]

                # Split target data into alignment and test sets
            align_indices = []
            test_indices = []
            for cls in classes:
                cls_indices = np.where(y_target == cls)[0]
                np.random.shuffle(cls_indices)
                align_indices.extend(cls_indices[:align_per_class])
                test_indices.extend(cls_indices[align_per_class:])
            
            X_target_align = X_target[align_indices]  # Shape: (56, 22, 1001)
            y_target_align = y_target[align_indices]  # Shape: (56,)
            X_target_test = X_target[test_indices]    # Shape: (232, 22, 1001)
            y_target_test = y_target[test_indices]    # Shape: (232,)


            print("Source labels:", np.unique(y_source, return_counts=True))
            print("Target test labels:", np.unique(y_target_test, return_counts=True))
            print("Target align labels:", np.unique(y_target_align, return_counts=True))

            # Compute covariance matrices
            n_samples = X_source.shape[2]  # 1001
            print(n_samples)

            cov_estimator = Covariances(estimator='scm')  # Sample Covariance Matrix estimator
            C_source = cov_estimator.fit_transform(X_source)          # Shape: (288, 22, 22)
            C_target_align = cov_estimator.fit_transform(X_target_align)  # Shape: (56, 22, 22)
            C_target_test = cov_estimator.fit_transform(X_target_test)    # Shape: (232, 22, 22)

            # After computing C_source, check symmetry
            symmetry_error = np.max(np.abs(C_source - np.transpose(C_source, (0, 2, 1))))
            print(f"Symmetry error: {symmetry_error}")

            # # Compute covariance matrices with regularization
            M_source = mean_logeuclid(C_source)        # Shape: (22, 22)
            M_target_align = mean_logeuclid(C_target_align)
            M_target_test = mean_logeuclid(C_target_test)

            S_source = tangent_space(C_source, M_source, metric='logeuclid')          # Shape: (288, 253)
            S_target_align = tangent_space(C_target_align, M_target_align, metric='logeuclid')
            S_target_test = tangent_space(C_target_test, M_target_test, metric='logeuclid')
            
            # # Rescale to average norm of 1
            average_norm_source = np.mean(np.linalg.norm(S_source, axis=1))
            print(average_norm_source)
            bar_s_source = S_source  / average_norm_source  # Shape: (288, 253)

            average_norm_target_align = np.mean(np.linalg.norm(S_target_align, axis=1))
            bar_t_target_align = S_target_align / average_norm_target_align   # Shape: (56, 253)

            average_norm_target_test = np.mean(np.linalg.norm(S_target_test, axis=1))
            bar_t_target_test = S_target_test / average_norm_target_test  # Shape: (232, 253)

            # # Compute class means for alignment
            bar_s_k = [np.mean(bar_s_source[y_source == k], axis=0) for k in classes]  # List of 2 vectors, each (253,)
            bar_t_k = [np.mean(bar_t_target_align[y_target_align == k], axis=0) for k in classes]  # List of 2 vectors, each (253,)


            # # Stack class means into matrices
            bar_S = np.column_stack(bar_s_k)  # Shape: (253, 2)
            bar_T = np.column_stack(bar_t_k)  # Shape: (253, 2)


            # # Compute cross-product matrix
            C_st = bar_S @ bar_T.T  # Shape: (253, 253)

            # C_st, bar_S, bar_T = compute_enhanced_cst_pca(bar_s_source, bar_t_target_align, y_source, y_target_align, classes, n_clusters=3)

            # # print(np.isnan(C_st).sum(), np.isinf(C_st).sum())  # Check for numerical issues

            # # PCA Augmentation Step (as per the paper)
            # # 1. Perform SVD on C_st
            U, D, Vh = np.linalg.svd(C_st, full_matrices=False)  # U: (253, 253), D: (253,), Vh: (253, 253)
            # U contains left singular vectors, D contains singular values, Vh is V^T (right singular vectors transposed)

            cumsum_D = np.cumsum(D)
            total_D = np.sum(D)
            N_v = np.searchsorted(cumsum_D, 0.999 * total_D) + 1
            if N_v == 0:
                N_v = 1  # Ensure at least one singular vector

            # Compute rotation matrix
            rotation_matrix = U[:, :N_v] @ Vh[:N_v, :]  # Shape: (253, 253)
            # Align target test vectors
            hat_t_test = bar_t_target_test @ rotation_matrix
            

            # # Train SVC on source data
            clf = SVC(kernel='linear')
            clf.fit(bar_s_source, y_source)

            # # Predict on aligned test data
            y_pred = clf.predict(hat_t_test)


            # # After prediction, check distribution of classes
            # # Compute accuracy
            acc = accuracy_score(y_target_test, y_pred)
            accuracies.append(acc)

        print('tmin: ', tmin, ' tmax: ',tmax)
        for subj_idx in range(len(train_active_X)):
            print(f"Subject {subj_idx+1} Test Accuracy: {accuracies[subj_idx]:.2f}")

        print(f"\nMean Cross-Validation Accuracy: {np.mean(accuracies):.2f} ± {np.std(accuracies):.2f}")
        

        
    

In [8]:
twofour_crosssession(2)

/tmp/ipykernel_16481/1414561254.py:27: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B01: Epoch data shape (120, 3, 751)


/tmp/ipykernel_16481/1414561254.py:27: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B02: Epoch data shape (120, 3, 751)


/tmp/ipykernel_16481/1414561254.py:27: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


...

['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B07: Epoch data shape (120, 3, 251)


/tmp/ipykernel_16481/1414561254.py:115: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B08: Epoch data shape (120, 3, 251)


/tmp/ipykernel_16481/1414561254.py:115: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B09: Epoch data shape (120, 3, 251)
Loaded data for 9 subjects.
Source labels: (array([0, 1]), array([60, 60]))
Target test labels: (array([0, 1]), array([48, 48]))
Target align labels: (array([0, 1]), array([12, 12]))
251
Symmetry error: 0.0
0.8623704419376996
Source labels: (array([0, 1]), array([60, 60]))
Target test labels: (array([0, 1]), array([48, 48]))
Target align labels: (array([0, 1]), array([12, 12]))
251
Symmetry error: 0.0
0.6304879965970647
Source labels: (array([0, 1]), array([60, 60]))
Target test labels: (array([0, 1]), array([48, 48]))
Target align labels: (array([0, 1]), array([12, 12]))
251
Symmetry error: 0.0
0.9857470042170065
Source labels: (array([0, 1]), array([60, 60]))
Target test labels: (array([0, 1]), array([58, 58]))
Target align labels: (array([0, 1]), array([12, 12]))
251
Symmetry error: 0.0
0.9707805267138075
Source labels: (array([0, 1]), array([60, 60]))
Target test labels: (